In [7]:
include("../src/naml.jl")
include("../test/wordnet.jl")

mkComponentwiseDistance (generic function with 1 method)

In [16]:
p = 5
prec = 20
K = PadicField(p, prec)

Field of 5-adic numbers

In [ ]:
# For testing. Creates the DAG matrix representation
# of a line with `n` nodes. Index of top node is `1`, and we go upwards.
function mk_line_matrix(n::Int64) 
    D = Matrix{Bool}(undef, n, n)
    for i in 1:n 
        for j in 1:n 
            D[i, j] = (j < i)
        end 
    end
    return D
end     

D = mk_line_matrix(10)

D = [false false false false false false; 
     true false false false false false; 
     true false false false false false; 
     true false false false false false;
     true true false false false false; 
     true true false false false false
     ]



6×6 Matrix{Bool}:
 0  0  0  0  0  0
 1  0  0  0  0  0
 1  0  0  0  0  0
 1  0  0  0  0  0
 1  1  0  0  0  0
 1  1  0  0  0  0

Here we're trying to embed a DAG in the disc space. We represent DAGs using a boolean matrix $D$, where $D_{ij}$ is true iff there is an edge $j \to i$. 

The main loss function we work with is
$$
    \mathcal{L} = \sum_{D_{ij} = 1} \log \frac{e ^ {- d(\theta_i, \theta_j)}}{\sum_{D_{ik} = 0} e ^ {- d(\theta_i, \theta_k)}}
$$
where $\theta_i$ is the polydisc embedding corresponding to node $i$. 
This is basically the same as is used in this paper: https://arxiv.org/pdf/1705.08039 (but working with the polydisc space rather than the hyperbolic disc!)

In [25]:
# not quite the radial sum in higher dimensions but that's fine
function radius_prod(p::ValuationPolydisc{S,T}) where S where T
    return Float64(prime(p)) ^ (- sum(p.radius))
end


# The loss function used in the Meta paper.
function init_distance_loss_new(D::Matrix{Bool}, K::Ring)
    # Quick and dirty implementation for now: we'll vectorise and optimise later.

    partial_sums::Vector{PolydiscFunction{PadicFieldElem}} = zeros(PolydiscFunction{PadicFieldElem}, size(D, 1))

    non_inj_penalty = 100


    function pairwise_distinct_count(p::ValuationPolydisc{PadicFieldElem, V}) where V
        comps = components(p)
        n = length(comps)
        total = 0
        for i in eachindex(p)
            for j in eachindex(p)
                if i != j && comps[i] == comps[j]
                    total += non_inj_penalty
                end
            end
        end
        return total
    end

    pairwise_distinct::PolydiscFunction{PadicFieldElem} = Lambda{PadicFieldElem}(pairwise_distinct_count)

    for i in axes(D, 1)
        den = Constant{PadicFieldElem}(0)
        num = Constant{PadicFieldElem}(0)
        for j in axes(D, 2)
            dist = mkComponentwiseDistance(i, j, K)
            if i == j
                continue
            end
            transformed_dist = comp(exp, -dist)
            if !D[i, j]
                den += transformed_dist
            else 
                num += transformed_dist
            end
        end
        if den == Constant{PadicFieldElem}(0)
            den += Constant{PadicFieldElem}(1)
        end
        if num != Constant{PadicFieldElem}(0)
            partial_sums[i] = comp(log, num / den)
        end
    end

    eval = pairwise_distinct
    eval += sum(partial_sums)
    eval = batch_evaluate_init(eval)
    beval = p_array -> map(eval, p_array)

    return Loss(beval, p -> 0)
end

function init_radial_loss(D::Matrix{Bool}, K::Ring)
    # Quick and dirty implementation for now: we'll vectorise and optimise later.

    partial_sums = Vector{PolydiscFunction{PadicFieldElem}}()

    non_inj_penalty = 500

    # TODO: let's not hardcode this:)
    pp = 5

    function pairwise_distinct_count(p::ValuationPolydisc{PadicFieldElem, V}) where V
        comps = components(p)
        n = length(comps)
        total = 0
        for i in eachindex(p)
            for j in eachindex(p)
                if i != j && comps[i] == comps[j]
                    total += non_inj_penalty
                end
            end
        end
        return total
    end

    function radial_penalty(i::Int64, j::Int64)
        function lam(p::ValuationPolydisc{PadicFieldElem, V}) where V
            comps = components(p)
            polydisc_join = join(comps[i], comps[j])
            output = exp(radius_prod(polydisc_join) - radius_prod(comps[j]))
            @assert output != 0
            return output
        end
        return lam
    end

    function normalisation(i::Int64, j::Int64)
        function lam(p::ValuationPolydisc{PadicFieldElem, V}) where V
            comps = components(p)
            output = exp(radius_prod(comps[j]) - radius_prod(comps[i])) 
            @assert output != 0
            return output
        end
        return lam
    end

    function second_summand(i::Int64, j::Int64)
        function lam(p::ValuationPolydisc{PadicFieldElem, V}) where V
            comps = components(p)
            polydisc_join = join(comps[i], comps[j])
            output = 1 * exp(-(radius_prod(polydisc_join) - max(radius_prod(comps[i]), radius_prod(comps[j]))))
            @assert output != 0
            return output
        end
        return lam
    end

    pairwise_distinct::PolydiscFunction{PadicFieldElem} = Lambda{PadicFieldElem}(pairwise_distinct_count)

    for i in axes(D, 1)
        for j in axes(D, 2)
            if D[i, j]
                push!(partial_sums, comp(x -> x, Lambda{PadicFieldElem}(radial_penalty(i, j)) / Lambda{PadicFieldElem}(normalisation(i, j))))
            else
                push!(partial_sums, Lambda{PadicFieldElem}(second_summand(i, j)))
            end 
        end
    end

    eval = pairwise_distinct
    eval = sum(partial_sums)
    eval = batch_evaluate_init(eval)
    beval = p_array -> map(eval, p_array)

    return Loss(beval, p -> 0)
end


## TODO: generic Gauss point constructor
function mk_initial_for(D::Matrix{Bool})
    return ValuationPolydisc(zeros(K, size(D, 1)), zeros(Int64, size(D, 1)))
end

ℓ_poly = init_rational_loss(D, K)
ℓ_dist = init_radial_loss(D, K)

next_branch = 1

# Here we don't want to force every coordinate to shrink! 
# The whole point being that which coordinates need to shrink more should be decided by the loss in this situation. 
config = (false, 1)

initial = mk_initial_for(D)

function check_embedding(p::ValuationPolydisc{PadicFieldElem, V}, D::Matrix{Bool}) where V
    comps = components(p)
    for i in eachindex(p) 
        for j in eachindex(p)
            if D[i, j]
                # then i should embed into j
                if !(comps[i] <= comps[j])
                    println("Failed at $i, $j")
                    return false
                end
            elseif i != j
                # then i should not embed into j
                if comps[i] <= comps[j]
                    println("Failed at $i, $j")
                    return false
                end
            end
        end
    end
    return true
end

# We optimise using Greedy descent
greedy_optim_with_poly::OptimSetup = greedy_descent_init(initial, ℓ_dist, next_branch, config)

OptimSetup{PadicFieldElem, Int64, Int64, Tuple{Bool, Int64}}(Loss(var"#init_radial_loss##100#init_radial_loss##101"(Core.Box(var"#batch_evaluate_init##29#batch_evaluate_init##30"{var"#lam#init_radial_loss##97"{Int64, Int64}, var"#batch_evaluate_init##29#batch_evaluate_init##30"{var"#lam#init_radial_loss##97"{Int64, Int64}, var"#batch_evaluate_init##29#batch_evaluate_init##30"{var"#lam#init_radial_loss##97"{Int64, Int64}, var"#batch_evaluate_init##29#batch_evaluate_init##30"{var"#lam#init_radial_loss##97"{Int64, Int64}, var"#batch_evaluate_init##29#batch_evaluate_init##30"{var"#eval#batch_evaluate_init##39"{PadicFieldElem, Comp{PadicFieldElem}, var"#batch_evaluate_init##35#batch_evaluate_init##36"{var"#lam#init_radial_loss##95"{Int64, Int64}, var"#lam#init_radial_loss##93"{Int64, Int64}}}, var"#batch_evaluate_init##29#batch_evaluate_init##30"{var"#eval#batch_evaluate_init##39"{PadicFieldElem, Comp{PadicFieldElem}, var"#batch_evaluate_init##35#batch_evaluate_init##36"{var"#lam#init_radia

In [26]:
optim_setup::OptimSetup = greedy_optim_with_poly

new_param, new_state = optim_setup.optimiser(
        optim_setup.loss,
        optim_setup.param,
        optim_setup.state,
        optim_setup.context
    )

println("")

# Please write a piece of code that logs the loss vector (i.e. the loss at each child node) at each step of the optimisation process into
# a separate file called loss_log.txt

N_epochs = 100
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_with_poly))
    #  ". Embedding correct so far: ", check_embedding(greedy_optim_with_poly.param, D)
    if check_embedding(greedy_optim_with_poly.param, D)
        break
    end
    step!(greedy_optim_with_poly)
end 
t2 = time()

println("Result of optimisation process is $(greedy_optim_with_poly.param)")

println("Embedding is correct: ", check_embedding(greedy_optim_with_poly.param, D))


Loss at epoch 1 is 36.0
Failed at 1, 2
Loss at epoch 2 is 34.89865792823444
Failed at 1, 2
Loss at epoch 3 is 32.69597378470333
Failed at 1, 2
Loss at epoch 4 is 29.942618605289432
Failed at 1, 2
Loss at epoch 5 is 26.08792135410998
Failed at 1, 2
Loss at epoch 6 is 25.110778139071332
Failed at 2, 5
Loss at epoch 7 is 24.71528201174171
Failed at 2, 5
Loss at epoch 8 is 24.3681175662816
Failed at 6, 2
Loss at epoch 9 is 24.03593717557106
Failed at 6, 2
Loss at epoch 10 is 23.570884628576298
Failed at 6, 2
Loss at epoch 11 is 23.499099357988616
Failed at 6, 2
Loss at epoch 12 is 23.436086620483337
Failed at 6, 2
Loss at epoch 13 is 23.375793592098745
Failed at 6, 2
Loss at epoch 14 is 23.29138335236034
Failed at 6, 2
Loss at epoch 15 is 23.277299901109785
Failed at 6, 2
Loss at epoch 16 is 23.26493752078365
Failed at 6, 2
Loss at epoch 17 is 23.253108716363865
Failed at 6, 2
Loss at epoch 18 is 23.236548390176186
Failed at 6, 2
Loss at epoch 19 is 23.233742499873472
Failed at 6, 2
Loss 

In [6]:
N_epochs = 1000
greedy_optim_with_distance::OptimSetup = greedy_descent_init(initial, ℓ_dist, next_branch, config)
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim_with_distance))
    step!(greedy_optim_with_distance)
end 
t2 = time()

println("Result of optimisation process is $(greedy_optim_with_distance.param)")

Loss at epoch 1 is 9.0
Loss at epoch 2 is 8.513417119032592
Loss at epoch 3 is 7.053668476130369
Loss at epoch 4 is 6.951363647604964
Loss at epoch 5 is 6.644449162028749
Loss at epoch 6 is 6.615096939134938
Loss at epoch 7 is 6.527040270453502
Loss at epoch 8 is 6.517729516554033
Loss at epoch 9 is 6.489797254855628
Loss at epoch 10 is 6.486744408791222
Loss at epoch 11 is 6.477585870598006
Loss at epoch 12 is 6.476573826115068
Loss at epoch 13 is 6.473537692666255
Loss at epoch 14 is 6.47320096104148
Loss at epoch 15 is 6.472190766167152
Loss at epoch 16 is 6.472078590705801
Loss at epoch 17 is 6.471742064321749
Loss at epoch 18 is 6.471704680099459
Loss at epoch 19 is 6.471592527432594
Loss at epoch 20 is 6.47158006686928
Loss at epoch 21 is 6.47154268517934
Loss at epoch 22 is 6.471538531752022
Loss at epoch 23 is 6.471526071470066
Loss at epoch 24 is 6.471524687004714
Loss at epoch 25 is 6.471520533608658
Loss at epoch 26 is 6.471520072121365
Loss at epoch 27 is 6.471518687659486
